# model-check template

The standard structure for a `model_checks/` notebook. Fill the four config cells — **SPACE**,
**TARGET**, **SEGMENTER**, **MODEL** — and the Summary cell runs the three gates and writes one row to
`results.md`. Rules of the house:

- **one segmentation method per notebook.** A different method (different boundaries, different
  algorithm) is a different notebook — not a second variant here.
- **the space changes in one place** (the SPACE cell).
- **the SEGMENTER cell owns its own segment-edge print** (a self-description of the method).
- **extra checks go below the Summary**, outside the template (a given segmenter may expose a knob,
  e.g. a cluster count, that other methods do not).

Copy this file, rename it `{space}_{target}_{segmenter}_{model}`, and edit the four cells.

In [ ]:
# --- SETUP -------------------------------------------------------------------
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import run_check, collect_bets, pooled_roi, roi_grouped, season_folds

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [ ]:
# --- SPACE (modifiable) — the one place that defines this check's space ------
# The running (point-in-time) season features that describe the match. This is the ONLY
# cell to change to move the whole check to another space.
FEATURES = [
    "hmt_season_goals_for_avg",
    "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg",
    "awt_season_goals_agst_avg",
]

In [ ]:
# --- TARGET (modifiable) -----------------------------------------------------
# OUTCOME is the 0/1 event we bet on. IMPLIED is the market-consensus implied probability
# (mrkt), used for both the profitability check and the ROI — one odds source, never mixed.
OUTCOME = "t_draw_flg"          # 1 when the draw happened
IMPLIED = "mrkt_draw_impl"      # implied prob = 1 / mrkt odds (profitability + betting)

In [ ]:
# --- ABT FILTERING -----------------------------------------------------------
# Keep matches where both teams have enough history for stable season averages; drop rows missing any
# feature or the target.
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}   "
      f"leagues: {sorted(matches['league'].unique())}")

In [ ]:
# --- SEGMENTER (modifiable) — KMeans on the running space --------------------
# The segmentation method IS this notebook's identity; a different method = a different notebook.
# Here: fit KMeans on the space features (train only), keep the clusters that beat the market on train,
# label each row by its kept cluster (NaN outside -> not bet). Boundaries and assignment both use the
# running (point-in-time) features. Contract: segmenter(train) -> labeler ; labeler(rows) -> Series.

# Keep the clusters/cells where the target happened MORE often than the market priced it, on the TRAIN
# fold only, so the held-out fold never influences the choice.
def keep_profitable(train, labels, min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    return set(stats.index[(stats["n"] >= min_matches) & (stats["won"] > stats["implied"])])

def kmeans_profitable_segmenter(features, n_clusters, min_matches=100):
    def segmenter(train):
        km = KMeans(n_clusters=n_clusters, random_state=0, n_init=10).fit(train[features].to_numpy())
        kept = keep_profitable(train, pd.Series(km.predict(train[features].to_numpy()), index=train.index), min_matches)
        def labeler(rows):
            c = pd.Series(km.predict(rows[features].to_numpy()), index=rows.index)
            return c.where(c.isin(kept))     # non-kept clusters -> NaN -> not bet
        return labeler
    return segmenter

K = 5   # cluster count (see the k-sweep extra check below)
segmenter = kmeans_profitable_segmenter(FEATURES, K)

# The segmenter's self-description: fit on all matches (in-sample) and show each cluster's edge — size,
# outcome rate vs implied, ROI, centre. In-sample, so a map of WHERE the edge sits, not gated evidence.
# KEEP marks clusters a training fold would keep (positive edge).
_km = KMeans(n_clusters=K, random_state=0, n_init=10).fit(matches[FEATURES].to_numpy())
_lab = _km.predict(matches[FEATURES].to_numpy())
print(f"segment edges (in-sample, k={K})   centre = [{'  '.join(FEATURES)}]")
print(" seg      n   rate%  impl%   edge     ROI   keep   centre")
for s in range(K):
    m = matches[_lab == s]
    dr, im = m[OUTCOME].mean(), m[IMPLIED].mean()
    c = _km.cluster_centers_[s]
    print(f" {s:>3} {len(m):>7}  {dr*100:5.1f}  {im*100:5.1f}  {(dr-im)*100:+5.1f}  "
          f"{pooled_roi(m[IMPLIED], m[OUTCOME].astype(int)):+6.3f}  {'KEEP' if dr > im else '    '}  "
          f"[{c[0]:.2f} {c[1]:.2f} {c[2]:.2f} {c[3]:.2f}]")

In [ ]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: any sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Summary
The three gates at a glance, and the one-line row written to `model_checks/results.md`.

In [ ]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="REPLACE_WITH_NOTEBOOK_NAME", space="SPACE", target="TARGET",
              abt_filter="major, gn>8, dropna", segmentation="KMeans k=5 (match, running, 4D)",
              segment_filter="train-profitable clusters")

## Extra checks — below the template
Everything above is the standard template (config cells + Summary). Cells below are notebook-specific
diagnostics, outside the template — they may use knobs a given segmenter has (e.g. a cluster count)
that other methods do not.